# SQLite Database — FinTech 590

Loads pool snapshots and historical data into a local SQLite database.

**Run order:** `defi_pipeline.ipynb` → `historical_data.ipynb` → this notebook.

**Schema:**
```
pools          — one row per pool (current snapshot)
pool_history   — one row per pool per day (historical TVL, APY, IL)
```

## 0. Imports

In [1]:
import sqlite3, pathlib
from datetime import date
import pandas as pd

DB_PATH          = pathlib.Path("data/defi_pools.db")
POOLS_PARQUET    = pathlib.Path("data/top_pools.parquet")
HISTORY_PARQUET  = pathlib.Path("data/pool_history.parquet")

print(f"Database : {DB_PATH}")
print(f"Pools    : {POOLS_PARQUET}")
print(f"History  : {HISTORY_PARQUET}")

Database : data\defi_pools.db
Pools    : data\top_pools.parquet
History  : data\pool_history.parquet


## 1. Create Tables

Uses `CREATE TABLE IF NOT EXISTS` — safe to re-run without wiping data.

In [2]:
con = sqlite3.connect(DB_PATH)
cur = con.cursor()

cur.execute("""
    CREATE TABLE IF NOT EXISTS pools (
        address            TEXT PRIMARY KEY,
        token0             TEXT,
        token1             TEXT,
        fee_tier           INTEGER,
        tvl_usd            REAL,
        volume_usd         REAL,
        etherscan_verified INTEGER,
        fetched_at         TEXT
    )
""")

cur.execute("""
    CREATE TABLE IF NOT EXISTS pool_history (
        address     TEXT,
        date        TEXT,
        tvl_usd     REAL,
        apy         REAL,
        apy_base    REAL,
        apy_base_7d REAL,
        il_7d       REAL,
        PRIMARY KEY (address, date),
        FOREIGN KEY (address) REFERENCES pools(address)
    )
""")
con.commit()
print("Tables created (or already exist).")

# Print schema
for (t,) in cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall():
    print(f"\n  [{t}]")
    for col in cur.execute(f"PRAGMA table_info({t})").fetchall():
        print(f"    {col[1]:20s} {col[2]}" + (" <- PK" if col[5] else ""))

Tables created (or already exist).

  [pool_history]
    address              TEXT <- PK
    date                 TEXT <- PK
    tvl_usd              REAL
    apy                  REAL
    apy_base             REAL
    apy_base_7d          REAL
    il_7d                REAL

  [pools]
    address              TEXT
    llama_id             TEXT
    token0               TEXT
    token1               TEXT
    fee_tier             INTEGER
    tvl_usd              REAL
    volume_usd           REAL
    etherscan_verified   INTEGER
    fetched_at           TEXT


## 2. Load Pools

Uses `INSERT OR REPLACE` — re-running always reflects the latest snapshot.

In [3]:
pools_df = pd.read_parquet(POOLS_PARQUET)
pools_df["fetched_at"] = str(date.today())

pools_df[["address", "token0", "token1", "fee_tier",
           "tvl_usd", "volume_usd",
           "etherscan_verified", "fetched_at"]].to_sql(
    "pools", con, if_exists="replace", index=False
)
con.commit()

count = cur.execute("SELECT COUNT(*) FROM pools").fetchone()[0]
print(f"pools table: {count} rows loaded")
pd.read_sql(
    "SELECT address, token0, token1, fee_tier, tvl_usd, etherscan_verified FROM pools",
    con
)

pools table: 20 rows loaded


,address,token0,token1,fee_tier,tvl_usd,etherscan_verified
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,USDC,WETH,500,95980263.0,1
1,0x4e68Ccd3E89f51C3074ca5072bbAC773960dFa36,WETH,USDT,3000,64839313.0,1
2,0x4585FE77225b41b697C938B018E2Ac67Ac5a20c0,WBTC,WETH,500,46502714.0,1
3,0xCBCdF9626bC03E24f779434178A73a0B4bad62eD,WBTC,WETH,3000,44304648.0,1
4,0xe8f7c89C5eFa061e340f2d2F206EC78FD8f7e124,WBTC,CBBTC,100,29150519.0,1
5,0xC5c134A1f112efA96003f8559Dba6fAC0BA77692,WHITE,WETH,100,28821407.0,1
6,0x99ac8cA7087fA4A2A1FB6357269965A2014ABc35,WBTC,USDC,3000,27601914.0,1
7,0x3416cF6C708Da44DB2624D63ea0AAef7113527C6,USDC,USDT,100,26885479.0,1
8,0x9Db9e0e53058C89e5B94e29621a205198648425B,WBTC,USDT,3000,26475304.0,1
9,0xbAFeAd7c60Ea473758ED6c6021505E8BBd7e8E5d,AUSD,USDC,100,25006727.0,1


## 3. Load Historical Data

Uses `INSERT OR IGNORE` — re-running skips existing (address, date) pairs so no duplicates are added.

In [4]:
history_df = pd.read_parquet(HISTORY_PARQUET)
history_df["date"] = history_df["date"].dt.strftime("%Y-%m-%d")

rows = history_df[["address", "date", "tvl_usd", "apy",
                    "apy_base", "apy_base_7d", "il_7d"]].values.tolist()

cur.executemany("""
    INSERT OR IGNORE INTO pool_history
        (address, date, tvl_usd, apy, apy_base, apy_base_7d, il_7d)
    VALUES (?, ?, ?, ?, ?, ?, ?)
""", rows)
con.commit()

count     = cur.execute("SELECT COUNT(*) FROM pool_history").fetchone()[0]
date_range = cur.execute("SELECT MIN(date), MAX(date) FROM pool_history").fetchone()
print(f"pool_history table: {count:,} rows")
print(f"Date range        : {date_range[0]}  ->  {date_range[1]}")

pool_history table: 20,003 rows
Date range        : 2022-03-27  ->  2026-03-26


## 4. Example Queries

In [5]:
# Top pools by average TVL across all history
print("Top pools by average historical TVL")
pd.read_sql("""
    SELECT p.token0, p.token1, p.fee_tier,
           ROUND(AVG(h.tvl_usd) / 1e6, 2) AS avg_tvl_usd_m,
           ROUND(MAX(h.tvl_usd) / 1e6, 2) AS peak_tvl_usd_m,
           COUNT(h.date)                   AS days_tracked
    FROM pool_history h
    JOIN pools p ON h.address = p.address
    GROUP BY h.address
    ORDER BY avg_tvl_usd_m DESC
""", con)

Top pools by average historical TVL


,token0,token1,fee_tier,avg_tvl_usd_m,peak_tvl_usd_m,days_tracked
0,WBTC,WETH,3000,187.72,434.86,1453
1,USDC,WETH,500,163.57,345.28,1433
2,USDC,WETH,3000,111.72,425.41,1433
3,WETH,USDT,3000,84.25,223.29,1453
4,WBTC,WETH,500,69.87,142.31,1453
5,USDC,USDT,100,62.69,222.27,1433
6,WBTC,USDC,3000,59.46,174.63,1433
7,WBTC,CBBTC,100,43.84,69.92,528
8,UNI,WETH,3000,27.18,55.14,1453
9,WBTC,USDT,3000,25.88,102.88,1453


In [6]:
# Average base APY by pool — last 90 days
print("Average base APY — last 90 days")
pd.read_sql("""
    SELECT p.token0, p.token1, p.fee_tier,
           ROUND(AVG(h.apy_base), 2) AS avg_apy_base_pct,
           ROUND(MAX(h.apy_base), 2) AS max_apy_base_pct
    FROM pool_history h
    JOIN pools p ON h.address = p.address
    WHERE h.date >= date('now', '-90 days')
      AND h.apy_base IS NOT NULL
    GROUP BY h.address
    ORDER BY avg_apy_base_pct DESC
""", con)

Average base APY — last 90 days


,token0,token1,fee_tier,avg_apy_base_pct,max_apy_base_pct
0,WETH,USDT,3000,48.15,408.86
1,USDC,WETH,500,38.95,345.15
2,WBTC,USDT,500,28.09,137.94
3,USDC,WETH,3000,27.08,193.45
4,WBTC,WETH,500,17.70,135.42
5,WBTC,USDC,3000,17.32,119.47
6,LINK,WETH,3000,16.48,94.64
7,WBTC,USDT,3000,16.04,89.79
8,WBTC,WETH,3000,8.44,87.34
9,UNI,WETH,3000,7.34,63.86


In [7]:
# Monthly average TVL for top 5 pools
print("Monthly average TVL — top 5 pools")
pd.read_sql("""
    SELECT p.token0 || '/' || p.token1  AS pair,
           strftime('%Y-%m', h.date)    AS month,
           ROUND(AVG(h.tvl_usd) / 1e6, 2) AS avg_tvl_usd_m
    FROM pool_history h
    JOIN pools p ON h.address = p.address
    WHERE p.address IN (
        SELECT address FROM pools ORDER BY tvl_usd DESC LIMIT 5
    )
    GROUP BY p.address, month
    ORDER BY month DESC, avg_tvl_usd_m DESC
    LIMIT 30
""", con)

Monthly average TVL — top 5 pools


,pair,month,avg_tvl_usd_m
0,USDC/WETH,2026-03,94.27
1,WETH/USDT,2026-03,61.55
2,WBTC/WETH,2026-03,47.68
3,WBTC/WETH,2026-03,46.62
4,WBTC/CBBTC,2026-03,29.76
5,USDC/WETH,2026-02,78.85
6,WETH/USDT,2026-02,63.97
7,WBTC/WETH,2026-02,50.71
8,WBTC/WETH,2026-02,37.90
9,WBTC/CBBTC,2026-02,28.61


In [8]:
con.close()
print(f"Database saved to {DB_PATH}  ({DB_PATH.stat().st_size / 1024:.0f} KB)")

Database saved to data\defi_pools.db  (3208 KB)
